# Raceline Optimization — two solvers, one scorer

Builds a minimum-curvature raceline from `centerline_full.csv` two ways, scores
both under the **simulator's own physics**, and exports both. The solvers, the
scorer and the physics live in `optimize_raceline.py`; this notebook is the
visual layer over it. Run that script for the production ladder
(`raceline_a4.0 / 4.5 / 4.9.csv`).

| Method | Solver | Notes |
|---|---|---|
| **A — scipy** | `scipy.optimize.lsq_linear` | Hand-rolled bound-constrained least squares on lateral offsets. Converges in ~10 passes. |
| **B — TUM** | `tph.opt_min_curv` | TUM's QP with `kappa_bound` hard and the car width inside the solve, iterated with a shrinking trust region. |

Both push each point sideways along its normal to minimise curvature, subject to
staying inside a corridor that is **re-measured from the map at every pass**.
On this track they agree within 0.2 s; A is smoother.

## Where the numbers come from

Every limit is from `VEHICLE_MODEL.md`, which was read out of the simulator's
Unity source: the tire's robust lateral limit is the friction-curve asymptote,
4.90 m/s² (the profile uses 4.5); longitudinal 4.55 both ways; drag is linear
0.273·v and is fitted for `tph`. The earlier 6.0 / 5.0 / quadratic-0.05 were
guesses, and they were what made the exported lines undriveable.

## Three bugs this notebook used to have (2026-09-03 review)

- **Method B swapped the width columns.** tph's `reftrack` is
  `[x, y, w_tr_right, w_tr_left]`; the direction of tph's normal only fixes the
  sign of alpha, not which wall is on the right. The old TUM line broke the true
  corridor at 30 of 276 points and had −0.023 m of body clearance.
- **Method A linearised around the iterate but applied the offsets from the
  centerline**, so iterations 1–7 cycled with period 3 and never improved.
- **The scorer's physics** was a guess (see above), and exported widths were
  the centerline's rather than the line's.

`tph.iqp_handler` still never returns here (its linearisation error sits at ~4.5
on 1 m radii); the trust-region loop in `solve_tum` is the bounded replacement.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import yaml

RACELINE_DIR = Path.home() / "Documents/roboracer/raceline"
MAP = Path.home() / "Documents/roboracer/devkit_ws/src/racer_mapping/maps/track_clean"
sys.path.insert(0, str(RACELINE_DIR))

# One source of truth for the car, the corridor and the solvers. Every number in
# there was read out of the simulator's source -- see VEHICLE_MODEL.md.
from optimize_raceline import (PHYS, ProfileLimits, TrackMap, extract_centerline,
                               export_centerline, export_csv, load_xy, print_table,
                               score_line, solve_scipy, solve_tum)

SAFETY_MARGIN = 0.15        # [m] body-to-wall, on top of half the car width. A wall touch is a respawn.
MAX_ITERS     = 15

# Speed model: the robust (asymptote) limits, VEHICLE_MODEL.md §3.2 / §5.1.
# a_lat 4.9 is the hard ceiling, 4.0 the rung that always held on the car.
LIM = ProfileLimits(a_lat=4.5, v_max=8.0)
KAPPA_CAR = PHYS.kappa_car
tm = TrackMap(MAP)
print(f"a_lat {LIM.a_lat}  a_long {LIM.a_long:.2f}  drag 0.273 v ≈ {LIM.drag_coeff():.3f} v²/m  "
      f"kappa_car {KAPPA_CAR:.2f} 1/m")

## 1. Centerline

In [ ]:
cl_path = RACELINE_DIR / "centerline_full.csv"
if not cl_path.exists():                      # fresh start: rebuild from the map
    cx, cy = extract_centerline(tm, 400)
    export_centerline(cx, cy, tm, RACELINE_DIR, LIM)
    print("centerline rebuilt from the map")
cx, cy = load_xy(cl_path)
D = np.loadtxt(cl_path, delimiter=",")
w_right, w_left = D[:, 5], D[:, 6]
N = len(cx)

print(f"{N} points")
print(f"corridor after {SAFETY_MARGIN} m margin: "
      f"min {(w_left + w_right).min() - 2 * SAFETY_MARGIN:.2f} m, "
      f"median {np.median(w_left + w_right) - 2 * SAFETY_MARGIN:.2f} m")

## 2. The scorer

**One function judges every line.** This is the part that makes the comparison
meaningful: curvature is re-derived the same way (tph splines) and the velocity
profile uses the same g-g diagram, regardless of which solver produced the points.

An earlier version of this notebook scored the two methods with different
velocity models and reported a difference that was mostly the model, not the line.

In [ ]:
def score(x, y, label=""):
    """Curvature, velocity profile, lap time, wall margin and steering rate for any
    closed line -- the same splines and the same physics for every candidate."""
    return score_line(x, y, tm, LIM, PHYS, label)


base = score(cx, cy, "centerline")
print(f"centerline: {base['length']:.2f} m, |k|max {base['kmax']:.3f}, lap {base['t']:.3f} s, "
      f"body margin {base['body_margin']:+.3f} m")

## 3. Method A — scipy

Each point moves to $p_i = q_i + \alpha_i n_i$, where $q$ is the **current
iterate** and $n$ its left normal. For uniform spacing, curvature is proportional
to the second difference $p_{i-1}-2p_i+p_{i+1}$, which is linear in $\alpha$, so
each pass is bound-constrained least squares. The bounds are the corridor
ray-cast from the map at $q$, minus half the car and the safety margin.

The step is applied from $q$ along $n$, then re-sampled to uniform arc length
*without smoothing*, so the corridor measured at the start of the pass still
holds at the end. $\lambda\,\|S\alpha\|^2$ only damps the step: at the fixed point
$\alpha=0$ and the line is a stationary point of the pure curvature objective.
Measured: $\lambda$ from 0.5 to 4 changes the lap by under 0.01 s.

In [ ]:
print("method A (scipy):")
res_a, hist_a = solve_scipy(tm, cx, cy, LIM, SAFETY_MARGIN, lam=2.0, iters=MAX_ITERS)
res_a["label"] = "A-scipy"

## 4. Method B — TUM

`tph.opt_min_curv` is the real QP: `kappa_bound` is a hard constraint and
`w_veh` keeps the *car* inside the walls rather than its centre point. The
reference track is `[x, y, w_right, w_left]` in **tph's column order**, widths
ray-cast from the map at the current line.

The QP linearises curvature around the reference. On 1 m radii a 0.9 m move
makes that linearisation meaningless (the first unconstrained pass gave
|k|max 2.7 and 8.7 rad/s of steering), so each pass is capped by a trust region
that shrinks geometrically from 0.15 m: big moves first, then refinement.

In [ ]:
print("method B (TUM):")
res_b, hist_b = solve_tum(tm, cx, cy, LIM, SAFETY_MARGIN, iters=MAX_ITERS)
res_b["label"] = "B-TUM"

## 5. Head to head

In [ ]:
rows = [base, res_a, res_b]
print_table(rows, SAFETY_MARGIN)

winner = min(res_a, res_b, key=lambda r: r["t"])
gap = abs(res_a["t"] - res_b["t"])
print(f"\nfaster here: {winner['label']} by {gap:.3f} s ({100 * gap / base['t']:.1f}% of a lap). "
      f"Differences under 0.5% are below what the physics resolves; prefer the smoother line.")

## 6. Each line on its own

Speed-coloured, one panel per line, on a **shared colour scale** so the three are
directly comparable. Red is slow, green is fast -- the corners where a line has
to give up speed are the ones costing lap time.

In [ ]:
meta = yaml.safe_load(open(MAP.with_suffix(".yaml")))
res_m, (ox, oy, _) = meta["resolution"], meta["origin"]
with open(MAP.with_suffix(".pgm"), "rb") as f:
    f.readline(); Wm, Hm = map(int, f.readline().split()); f.readline()
    img = np.frombuffer(f.read(Wm * Hm), dtype=np.uint8).reshape(Hm, Wm)
rgb = np.zeros((Hm, Wm, 3), np.uint8)
rgb[img == 0] = (25, 25, 25); rgb[img >= 254] = (255, 255, 255); rgb[img == 205] = (160, 160, 160)
extent = [ox, ox + Wm * res_m, oy, oy + Hm * res_m]
close = lambda a: np.append(a, a[0])

rows = [base, res_a, res_b]
vmin = min(r["vx"].min() for r in rows)
vmax = max(r["vx"].max() for r in rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 13))
for ax, r in zip(axes, rows):
    ax.imshow(rgb, extent=extent, origin="upper")
    sc = ax.scatter(r["x"], r["y"], c=r["vx"], cmap="RdYlGn",
                    vmin=vmin, vmax=vmax, s=7, zorder=3)
    ax.set_title(f"{r['label']}\n{r['t']:.3f} s   {r['length']:.2f} m\n"
                 f"|k|max {r['kmax']:.3f}", fontsize=10)
    ax.set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
fig.colorbar(sc, ax=axes, label="speed [m/s]", fraction=0.03, pad=0.02)
plt.show()

## 7. Side by side

The three overlaid, plus curvature and speed against lap fraction. Lap fraction
rather than metres, because the lines are different lengths -- this puts the same
corner at the same x on all three traces.

In [ ]:
fig = plt.figure(figsize=(14, 11))

axm = fig.add_subplot(1, 2, 1)
axm.imshow(rgb, extent=extent, origin="upper")
axm.plot(close(base["x"]), close(base["y"]), "--", color="#666", lw=1.0,
         label=f"centerline  {base['t']:.2f}s")
axm.plot(close(res_a["x"]), close(res_a["y"]), "-", color="#d62728", lw=1.7,
         label=f"A scipy  {res_a['t']:.2f}s")
axm.plot(close(res_b["x"]), close(res_b["y"]), "-", color="#1f77b4", lw=1.7,
         label=f"B TUM  {res_b['t']:.2f}s")
axm.legend(fontsize=9, loc="upper right")
axm.set_xlabel("x [m]"); axm.set_ylabel("y [m]"); axm.set_title("all three")

a1 = fig.add_subplot(2, 2, 2)
for r, c in ((base, "#888"), (res_a, "#d62728"), (res_b, "#1f77b4")):
    a1.plot(np.linspace(0, 1, len(r["kappa"])), r["kappa"], color=c, lw=1.1, label=r["label"])
a1.axhline(KAPPA_CAR, color="k", ls=":", lw=1, label="car limit")
a1.axhline(-KAPPA_CAR, color="k", ls=":", lw=1)
a1.set_ylabel("curvature [1/m]"); a1.set_xlabel("lap fraction")
a1.legend(fontsize=7); a1.grid(alpha=0.3); a1.set_title("curvature")

a2 = fig.add_subplot(2, 2, 4)
for r, c in ((base, "#888"), (res_a, "#d62728"), (res_b, "#1f77b4")):
    a2.plot(np.linspace(0, 1, len(r["vx"])), r["vx"], color=c, lw=1.1, label=r["label"])
a2.set_ylabel("speed [m/s]"); a2.set_xlabel("lap fraction")
a2.legend(fontsize=7); a2.grid(alpha=0.3); a2.set_title("velocity profile")

plt.tight_layout(); plt.show()

## 8. Export both

Both in the 8-column layout the follower reads. Width columns are ray-cast from
the map **at the exported line**, not copied from the centerline; heading is a
closed-loop central difference, so there is no seam.

In [ ]:
def export(r, filename):
    out = export_csv(RACELINE_DIR / filename, r, tm)
    print(f"{r['label']:<10} -> {filename}  ({r['n']} points, {r['t']:.3f} s)")
    return out


# The baseline gets a speed column too, so the centerline is raced under the
# same speed model as the racelines.
path_base = export(base, "centerline_speed.csv")
path_a = export(res_a, "raceline_scipy.csv")
path_b = export(res_b, "raceline_tum.csv")

# The validated ladder (same geometry, a_lat 4.0 / 4.5 / 4.9) is written by
#     python optimize_raceline.py
print("\nrun either:")
for p in (path_base, path_a, path_b):
    print(f"  ros2 launch racer_bringup race.launch.py path_csv:={p}")

## Tuning

| Knob | Where | Effect |
|---|---|---|
| `SAFETY_MARGIN` | both | Body-to-wall margin. Each 5 cm costs ~0.08 s; a touch costs a respawn. |
| `LIM.a_lat` | scorer | **The one that matters.** 4.90 is the tire's asymptote; 4.5 default; 4.0 always held. |
| `LIM.a_long` | scorer | 4.55 is held at any slip (wheels may spin at exits); ~3.5 keeps encoder overread under 10 %. |
| `lam` | A | Step damping only. 0.5–4 gives the same line. |
| `step_max` | B | First-pass trust region, shrinks ×0.8 per pass. |
| `stepsize` | B | Raceline resolution. Coarser inflates measured curvature. |

## Next

1. Race `raceline_a4.0.csv`, then `a4.5`, then `a4.9`; the first that runs wide
   gives the real usable lateral limit.
2. Measure the undocumented curve shapes as described in `VEHICLE_MODEL.md` §6.
3. Slip-aware throttle in `pure_pursuit` (§5.3 there) is what turns the peak
   limits into usable ones.